# Project 1: Dynamic Pricing Predictor for Ride-Sharing Apps
## Milestone 1: Data Understanding & Feature Engineering Pipeline
**Author:** Kamaljit  
**Track:** AI & ML Minor Project  
**Platform:** Nyeras LMS  

---

### Executive Summary
Dynamic pricing algorithms dynamically adjust fares in real-time according to supply and demand imbalances, traffic congestion, adverse weather conditions, and spatial-temporal trends.  

In this milestone, we:
1. Ingest and explore an extensive ride-sharing dataset modeled after Uber/Lyft operations.
2. Engineer four core feature pillars:
   - **Time-of-day features** (cyclical trigonometric hour encodings, rush-hour indicators)
   - **Distance & spatial features** (log-distance, speed estimation, congestion flags, airport routes)
   - **Weather impact features** (weather severity index, precipitation indicators)
   - **Demand-supply dynamics** (demand-supply ratio, surge flags, interaction terms)
3. Prepare clean, high-signal matrices for regression modeling in Milestone 2.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment initialized successfully.')

### 1. Ingesting and Exploring the Raw Dataset

In [2]:
data_path = os.path.join('..', 'data', 'raw', 'rides_dataset.csv')
df = pd.read_csv(data_path)
print(f'Total records: {df.shape[0]:,}, Features: {df.shape[1]}')
df.head()

In [3]:
# Checking data types, non-null values, and schema integrity
print('Dataset Info:')
print(df.info())
print('\nMissing Values Count:')
print(df.isnull().sum())

In [4]:
# Key numerical feature distribution
df[['distance_km', 'duration_min', 'active_drivers', 'ride_requests', 'demand_supply_ratio', 'surge_multiplier', 'price']].describe()

### 2. Feature Engineering Pillar 1: Time-of-Day Features
Standard hour numbers (0 to 23) create an artificial discontinuity between 23:00 and 00:00. We map `hour` to cyclical sine and cosine coordinates:  
$$\text{hour}_{\sin} = \sin\left(\frac{2\pi \cdot \text{hour}}{24}\right), \quad \text{hour}_{\cos} = \cos\left(\frac{2\pi \cdot \text{hour}}{24}\right)$$
We also build explicit business rush-hour flags.

In [5]:
df['datetime'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['datetime'].dt.hour
df['day_of_week_num'] = df['datetime'].dt.dayofweek

# Cyclical encodings
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)

# Rush hours (Weekdays 07:00-09:00 and 17:00-20:00)
df['is_morning_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 9) & (df['is_weekend'] == 0)).astype(int)
df['is_evening_rush'] = ((df['hour'] >= 17) & (df['hour'] <= 20) & (df['is_weekend'] == 0)).astype(int)
df['is_rush_hour'] = df['is_morning_rush'] | df['is_evening_rush']

print('Time-of-day features engineered successfully.')
df[['hour', 'hour_sin', 'hour_cos', 'is_morning_rush', 'is_evening_rush', 'is_rush_hour']].head()

### 3. Feature Engineering Pillar 2: Distance & Spatial Features
- `log_distance`: Handles long-tailed trip distributions
- `distance_squared`: Captures non-linear wear-and-tear and driver repositioning overheads
- `est_speed_kmh`: Distance over time to detect traffic jams
- `is_airport_trip`: Detects flat toll / premium airport tariffs

In [6]:
df['log_distance'] = np.log1p(df['distance_km'])
df['distance_squared'] = df['distance_km'] ** 2
df['est_speed_kmh'] = np.where(df['duration_min'] > 0, df['distance_km'] / (df['duration_min'] / 60.0), 30.0)
df['is_traffic_congested'] = (df['est_speed_kmh'] < 20.0).astype(int)
df['is_airport_trip'] = ((df['pickup_location'] == 'International Airport') | (df['dropoff_location'] == 'International Airport')).astype(int)

print('Distance & spatial features engineered successfully.')
df[['distance_km', 'log_distance', 'est_speed_kmh', 'is_traffic_congested', 'is_airport_trip']].head()

### 4. Feature Engineering Pillar 3: Weather Features
Weather conditions alter passenger willingness to walk and reduce active driver supply. We construct an ordinal `weather_severity_index` and adverse precipitation flags.

In [7]:
severity_map = {'Clear': 0.0, 'Cloudy': 0.2, 'Fog': 0.5, 'Rain': 0.7, 'Heavy Rain': 1.2, 'Thunderstorm': 1.8}
df['weather_severity_index'] = df['weather_condition'].map(severity_map).fillna(0.0)
df['is_adverse_weather'] = ((df['precipitation_mm'] > 0.5) | (df['weather_condition'].isin(['Heavy Rain', 'Thunderstorm']))).astype(int)

print('Weather impact features engineered successfully.')
df[['weather_condition', 'precipitation_mm', 'weather_severity_index', 'is_adverse_weather']].head()

### 5. Feature Engineering Pillar 4: Demand & Supply Dynamics & Interaction Terms
- `demand_pressure_index`: Bounded demand/supply ratio
- `distance_x_surge`: Cross-interaction between distance and surge multiplier
- `demand_x_weather`: Compound surge pressure under bad weather

In [8]:
df['demand_pressure_index'] = np.clip(df['demand_supply_ratio'], 0.5, 3.5)
df['is_surge_active'] = (df['surge_multiplier'] > 1.0).astype(int)
df['distance_x_surge'] = df['distance_km'] * df['surge_multiplier']
df['demand_x_weather'] = df['demand_pressure_index'] * (1.0 + df['weather_severity_index'])
df['duration_x_rush'] = df['duration_min'] * (1.0 + df['is_rush_hour'] * 0.5)

print('Demand and interaction features engineered successfully.')
df[['demand_supply_ratio', 'surge_multiplier', 'distance_x_surge', 'demand_x_weather']].head()

### 6. Correlation Analysis & Feature Selection
Evaluating Pearson correlations with target `price`.

In [9]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_target = df[numeric_cols].corr()['price'].sort_values(ascending=False)
print('Top Correlated Features with Ride Price:')
print(corr_target.head(12))

plt.figure(figsize=(10, 6))
corr_target.drop('price').head(10).plot(kind='barh', color='#2980b9')
plt.title('Top 10 Feature Correlations with Final Ride Price', fontsize=12, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### 7. Saving the Final Engineered Dataset
The processed dataset is exported for model training in Milestone 2.

In [10]:
out_path = os.path.join('..', 'data', 'processed', 'features_engineered.csv')
df.to_csv(out_path, index=False)
print(f'Successfully saved processed dataset with {df.shape[1]} columns to {out_path}')